In [1]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 1: Setup
# Purpose: Initialize paths, device, imports for ablation study
# ============================================================

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm

PROJECT_ROOT = "/mnt/g/banglafake-detection"
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 256

print("Device:", device)

Device: cuda


In [3]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 2: Define Model Architectures (Baseline + Ablations + Full)
# ============================================================

class BanglaBERTOnly(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token (ELECTRA has no pooler)
        output = self.dropout(cls_output)
        return self.classifier(output)


class BanglaBERTWithCNN(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(cnn_channels, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        pooled = torch.mean(x, dim=2)
        return self.classifier(self.dropout(pooled))


class BanglaBERTWithAttention(nn.Module):
    def __init__(self, model_name, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.attention_projection = nn.Linear(self.bert.config.hidden_size, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(self.dropout(attended))


class BanglaBERTFullModel(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.attention_projection = nn.Linear(cnn_channels, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_channels, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        x = x.permute(0, 2, 1)
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(attended)


print("Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel")

Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel


In [5]:
# ============================================================
# Cell 3: Load Data, Tokenizer, Dataset, DataLoaders
# ============================================================

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class BanglaFakeNewsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = str(self.dataframe.iloc[idx]["text"])
        label = int(self.dataframe.iloc[idx]["label"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_loader = DataLoader(
    BanglaFakeNewsDataset(train_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=True, pin_memory=True
)
val_loader = DataLoader(
    BanglaFakeNewsDataset(val_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)
test_loader = DataLoader(
    BanglaFakeNewsDataset(test_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 2790 Val: 598 Test: 599


In [6]:
# ============================================================
# Cell 4: Train/Eval Helper Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0
    for batch in tqdm(loader, desc="Train", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def eval_split(model, loader, device):
    model.eval()
    preds, labels_all, probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return {
        "Accuracy": accuracy_score(labels_all, preds),
        "Precision": precision_score(labels_all, preds),
        "Recall": recall_score(labels_all, preds),
        "F1": f1_score(labels_all, preds),
        "ROC-AUC": roc_auc_score(labels_all, probs)
    }


print("Train/eval functions ready.")

Train/eval functions ready.


In [8]:
# ============================================================
# Cell 5: Train Ablation Models (BanglaBERT-only, +CNN, +Attention)
# Full train set (2790), same NUM_EPOCHS=3, best-val-F1 checkpoint
# ============================================================

NUM_EPOCHS = 3
ablation_results = {}

model_configs = {
    "BanglaBERT-only": lambda: BanglaBERTOnly(MODEL_NAME),
    "BanglaBERT+CNN": lambda: BanglaBERTWithCNN(MODEL_NAME),
    "BanglaBERT+Attention": lambda: BanglaBERTWithAttention(MODEL_NAME),
}

for name, ctor in model_configs.items():
    print(f"\n=== Training {name} ===")
    model = ctor().to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    best_val_f1, best_state = -1, None

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = eval_split(model, val_loader, device)
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_f1={val_metrics['F1']:.4f}")
        if val_metrics['F1'] > best_val_f1:
            best_val_f1 = val_metrics['F1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(MODELS_DIR, f"ablation_{name.replace('+','_').replace(' ','_')}.pt"))
    test_metrics = eval_split(model, test_loader, device)
    ablation_results[name] = test_metrics
    print(f"{name} TEST:", test_metrics)

    del model
    torch.cuda.empty_cache()

print("\nAblation training complete.")


=== Training BanglaBERT-only ===


Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 1: train_loss=0.1620 val_f1=0.9739


Train:   0%|          | 0/349 [00:00<?, ?it/s]

...

Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 3: train_loss=0.0230 val_f1=0.9867


Eval:   0%|          | 0/75 [00:00<?, ?it/s]

BanglaBERT+Attention TEST: {'Accuracy': 0.9782971619365609, 'Precision': 0.979933110367893, 'Recall': 0.9766666666666667, 'F1': 0.9782971619365609, 'ROC-AUC': 0.9935340022296544}

Ablation training complete.


In [9]:
# ============================================================
# Cell 6: Load Existing Full Model Checkpoint & Evaluate (sanity check)
# ============================================================

full_model = BanglaBERTFullModel(MODEL_NAME).to(device)
checkpoint_path = os.path.join(MODELS_DIR, "best_banglabert_cnn_attention.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
full_model.load_state_dict(checkpoint["model_state_dict"])

full_model_metrics = eval_split(full_model, test_loader, device)
ablation_results["BanglaBERT+CNN+Attention (Full)"] = full_model_metrics
print("Full Model TEST (should match Notebook 2 results ~0.9733 acc):", full_model_metrics)

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Full Model TEST (should match Notebook 2 results ~0.9733 acc): {'Accuracy': 0.9732888146911519, 'Precision': 0.9797297297297297, 'Recall': 0.9666666666666667, 'F1': 0.9731543624161074, 'ROC-AUC': 0.9926086956521739}


In [10]:
# ============================================================
# Cell 7: Final Ablation Comparison Table
# ============================================================

comparison_df = pd.DataFrame(ablation_results).T
comparison_df = comparison_df.round(4)
display(comparison_df)

output_path = os.path.join(REPORTS_DIR, "ablation_comparison_table.csv")
comparison_df.to_csv(output_path)
print("\nSaved:", output_path)

,Accuracy,Precision,Recall,F1,ROC-AUC
BanglaBERT-only,0.9833,0.9801,0.9867,0.9834,0.9950
BanglaBERT+CNN,0.9800,0.9706,0.9900,0.9802,0.9906
BanglaBERT+Attention,0.9783,0.9799,0.9767,0.9783,0.9935
BanglaBERT+CNN+Attention (Full),0.9733,0.9797,0.9667,0.9732,0.9926



Saved: /mnt/g/banglafake-detection/reports/ablation_comparison_table.csv


In [11]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 8: Reload All Models & Get Per-Sample Predictions
# ============================================================

from statsmodels.stats.contingency_tables import mcnemar

def get_predictions(model, loader, device):
    model.eval()
    preds, labels_all = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predict", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            logits = model(input_ids, attention_mask)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
    return np.array(preds), np.array(labels_all)

model_defs = {
    "BanglaBERT-only": (BanglaBERTOnly(MODEL_NAME), "ablation_BanglaBERT-only.pt"),
    "BanglaBERT+CNN": (BanglaBERTWithCNN(MODEL_NAME), "ablation_BanglaBERT_CNN.pt"),
    "BanglaBERT+Attention": (BanglaBERTWithAttention(MODEL_NAME), "ablation_BanglaBERT_Attention.pt"),
    "BanglaBERT+CNN+Attention (Full)": (BanglaBERTFullModel(MODEL_NAME), "best_banglabert_cnn_attention.pt"),
}

all_preds = {}
true_labels = None

for name, (model, ckpt_file) in model_defs.items():
    model = model.to(device)
    state = torch.load(os.path.join(MODELS_DIR, ckpt_file), map_location=device)
    model.load_state_dict(state)
    preds, labels = get_predictions(model, test_loader, device)
    all_preds[name] = preds
    if true_labels is None:
        true_labels = labels
    del model
    torch.cuda.empty_cache()

print("Predictions collected for all 4 models on", len(true_labels), "test samples")



ModuleNotFoundError: No module named 'statsmodels'

In [12]:
# ============================================================
# Cell 9: McNemar's Test — Baseline vs Each Variant
# Purpose: Check if performance differences are statistically significant
# ============================================================

correct = {name: (preds == true_labels).astype(int) for name, preds in all_preds.items()}

baseline_name = "BanglaBERT-only"
mcnemar_results = []

for name in all_preds:
    if name == baseline_name:
        continue
    b_correct = correct[baseline_name]
    v_correct = correct[name]

    # Contingency table: [both_correct, baseline_only] / [variant_only, both_wrong]
    n01 = np.sum((b_correct == 1) & (v_correct == 0))  # baseline right, variant wrong
    n10 = np.sum((b_correct == 0) & (v_correct == 1))  # variant right, baseline wrong
    n00 = np.sum((b_correct == 0) & (v_correct == 0))
    n11 = np.sum((b_correct == 1) & (v_correct == 1))

    table = [[n11, n01], [n10, n00]]
    result = mcnemar(table, exact=(n01 + n10 < 25), correction=True)

    mcnemar_results.append({
        "Comparison": f"{baseline_name} vs {name}",
        "Baseline_correct_only": n01,
        "Variant_correct_only": n10,
        "statistic": result.statistic,
        "p_value": result.pvalue,
        "Significant (p<0.05)": result.pvalue < 0.05
    })

mcnemar_df = pd.DataFrame(mcnemar_results)
display(mcnemar_df)

mcnemar_df.to_csv(os.path.join(REPORTS_DIR, "mcnemar_test_results.csv"), index=False)
print("\nSaved:", os.path.join(REPORTS_DIR, "mcnemar_test_results.csv"))

NameError: name 'all_preds' is not defined